# SocialLLM — Interlocutor Identity Selectivity in GPT-2
### Can individual neurons distinguish *who* is speaking?

**Pipeline:**
1. **Social interactions** — 5 personas × 4 tasks × 2 episodes  
2. **Activation extraction** — forward hooks on every transformer layer (residual + MLP)  
3. **Selectivity analysis** — ANOVA, Selectivity Index, d-prime, lifetime sparseness  
4. **Population decoding** — linear classifier across layers  
5. **RSA / geometry** — PCA population geometry + representational dissimilarity matrices  

Each section saves a figure to `results/figures/` with a printed caption.

## 0 · Configuration

In [3]:
# ── Edit these before running ──────────────────────────────────────────
MODEL          = "gpt2"       # "gpt2-xl" for the real experiment (needs GPU)
N_EPISODES     = 2            # per (persona × task) condition
DEVICE         = "cpu"        # "cuda" if GPU is available
TASKS          = ["qa", "debate", "story", "problem_solving"]
FDR_ALPHA      = 0.05
AGGREGATION    = "mean"       # collapse token dim: "mean" | "last" | "max"
# ────────────────────────────────────────────────────────────────────────

import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

DATA_DIR         = "../data"
ACTIVATIONS_H5   = f"{DATA_DIR}/activations/activations_{MODEL.replace('-','_')}.h5"
INTERACTIONS_DIR = f"{DATA_DIR}/interactions"
FIGURES_DIR      = "../results/figures"

for d in [FIGURES_DIR, INTERACTIONS_DIR, f"{DATA_DIR}/activations"]:
    os.makedirs(d, exist_ok=True)

print(f"Model : {MODEL}")
print(f"Device: {DEVICE}")
print(f"Conditions: 5 personas × {len(TASKS)} tasks × {N_EPISODES} episodes = "
      f"{5 * len(TASKS) * N_EPISODES} interactions")

Model : gpt2
Device: cpu
Conditions: 5 personas × 4 tasks × 2 episodes = 40 interactions


## 1 · Imports

In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.social.agents    import LLMAgent, TargetAgent, make_interlocutor_configs, make_target_config
from src.social.environment import InteractionDatasetRunner
from src.social.tasks     import TASK_REGISTRY
from src.activation.extractor import ActivationDataset, load_layer, list_layers
from src.analysis.selectivity import run_full_analysis
from src.analysis.decoding    import decode_layer, pca_population_geometry, compute_rdm

print(f"PyTorch {torch.__version__} | device={DEVICE}")

PyTorch 2.12.0 | device=cpu


## 2 · Social Interactions

GPT-2 (target) reads messages produced by 5 persona-variants of *itself*.  
On each reading pass the target's hidden-unit activations are captured via forward hooks.

| Persona | Style |
|---|---|
| curious | Follow-up questions, wonder |
| formal | Numbered points, academic tone |
| creative | Metaphors, storytelling |
| concise | ≤2 sentences, blunt |
| skeptical | Devil's advocate, counter-arguments |

In [5]:
# ── Load models ──────────────────────────────────────────────────────────
print(f"Loading target ({MODEL})...")
target = TargetAgent(make_target_config(device=DEVICE, model_name=MODEL))
print("Target ready.")

interlocutor_cfgs = make_interlocutor_configs(device=DEVICE, model_name=MODEL)
interlocutors     = [LLMAgent(cfg) for cfg in interlocutor_cfgs]
print(f"Interlocutors: {[a.agent_id for a in interlocutors]}")

Loading target (gpt2)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7029.61it/s]


Target ready.


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 14624.63it/s]

Interlocutors: ['persona_curious', 'persona_formal', 'persona_creative', 'persona_concise', 'persona_skeptical']


In [6]:
# ── Run interactions ─────────────────────────────────────────────────────
tasks = [TASK_REGISTRY[t] for t in TASKS]

runner = InteractionDatasetRunner(
    target=target,
    interlocutors=interlocutors,
    tasks=tasks,
    output_dir=INTERACTIONS_DIR,
    n_episodes_per_condition=N_EPISODES,
    seed=42,
)

print(f"Running {len(interlocutors)} personas × {len(tasks)} tasks × {N_EPISODES} ep ...")
interactions = runner.run_all(verbose=True)
print(f"\nTotal interactions collected: {len(interactions)}")

Running 5 personas × 4 tasks × 2 ep ...
[1/40] persona=persona_curious | task=qa | episode=1/2


: 

In [ ]:
# ── Build activation dataset (HDF5) ──────────────────────────────────────
ds = ActivationDataset(output_path=ACTIVATIONS_H5, aggregation=AGGREGATION)
ds.add_from_interactions(interactions)
ds.save()

layers = list_layers(ACTIVATIONS_H5)
data0, labels0, label_map = load_layer(ACTIVATIONS_H5, layers[0])

print(f"Saved {len(layers)} layers → {ACTIVATIONS_H5}")
print(f"First layer shape : {data0.shape}  (N_samples × hidden_dim)")
print(f"Label map         : {label_map}")

# Convenience lists used in all figures below
residual_layers = sorted([l for l in layers if "residual"   in l])
mlp_layers      = sorted([l for l in layers if "mlp_hidden" in l])
persona_names   = [label_map[str(i)] for i in range(len(label_map))]
persona_short   = [n.replace("persona_", "") for n in persona_names]
COLORS          = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]

NameError: name 'ActivationDataset' is not defined

: 

: 

## 3 · Neuroscience-Style Selectivity Analysis

Each hidden unit = one "neuron".  Each interlocutor persona = one stimulus condition.  
We ask: does this neuron respond differently to different personas?

| Metric | Formula | Interpretation |
|---|---|---|
| ANOVA F-stat | one-way across conditions | significance of tuning |
| Selectivity Index (SI) | (R_max − R_2nd) / (R_max + R_2nd) | 0 = flat, 1 = maximally selective |
| d-prime | (μ_best − μ_2nd) / σ_pooled | signal-detection-theory separation |
| Lifetime sparseness | Rolls & Tovee 1995 | breadth of tuning across all conditions |

In [ ]:
print("Running selectivity analysis across all layers...")
reports = run_full_analysis(ACTIVATIONS_H5, fdr_alpha=FDR_ALPHA)
print(f"Done — {len(reports)} layers analyzed.\n")

# Quick summary for a middle layer
mid = residual_layers[len(residual_layers)//2]
r   = reports[mid]
print(f"--- {mid} ---")
print(f"  Fraction selective (FDR {FDR_ALPHA}): {r['fraction_selective']:.3f}")
print(f"  Mean SI                            : {r['mean_si']:.3f}")
print(f"  Mean d-prime                       : {r['mean_dprime']:.3f}")
print(f"  Preferred-persona counts           : {r['preferred_condition_counts']}")

: 

: 

### Figure 1 — Fraction of persona-selective units per layer

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

frac_res = [reports[l]["fraction_selective"] for l in residual_layers]
frac_mlp = [reports[l]["fraction_selective"] for l in mlp_layers]
x = range(len(residual_layers))

ax.plot(x, frac_res, "o-",  color="steelblue", label="Residual stream", lw=2)
ax.plot(x, frac_mlp, "s--", color="coral",     label="MLP hidden",      lw=2)
ax.axhline(FDR_ALPHA, color="gray", ls=":", lw=1.5, label=f"FDR α={FDR_ALPHA}")
ax.set_xlabel("Layer index", fontsize=12)
ax.set_ylabel("Fraction selective", fontsize=12)
ax.set_title("Fraction of Persona-Selective Units per Layer", fontsize=14, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(range(len(residual_layers)))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig1_fraction_selective.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
CAPTION — Figure 1
Fraction of hidden units whose response to the 5 interlocutor personas is 
statistically significant (one-way ANOVA, Benjamini–Hochberg FDR α=0.05).
Blue circles: residual stream (transformer block output).
Red squares: MLP hidden units (post-GELU activations).
Values above the dashed line exceed the FDR threshold.
Layers where many units are selective encode interlocutor identity most strongly.
""")

: 

: 

### Figure 2 — Selectivity Index distribution per layer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, layer_set, color, label in [
    (axes[0], residual_layers, "steelblue", "Residual stream"),
    (axes[1], mlp_layers,      "coral",     "MLP hidden"),
]:
    si_data = [reports[l]["selectivity_indices"] for l in layer_set]
    bp = ax.boxplot(si_data, positions=range(len(layer_set)),
                    patch_artist=True,
                    medianprops=dict(color="black", lw=2),
                    boxprops=dict(facecolor=color, alpha=0.4),
                    flierprops=dict(marker=".", ms=2, alpha=0.3))
    ax.set_xlabel("Layer index", fontsize=11)
    ax.set_ylabel("Selectivity Index (SI)", fontsize=11)
    ax.set_title(f"SI Distribution — {label}", fontsize=12, fontweight="bold")
    ax.set_xticks(range(len(layer_set))); ax.set_xticklabels(range(len(layer_set)))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig2_selectivity_index.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
CAPTION — Figure 2
Distribution of Selectivity Index (SI) across transformer layers.
SI = (R_max − R_2nd) / (R_max + R_2nd), where R_max is the mean response to the
preferred persona and R_2nd to the second-best.
SI = 0: responds equally to all personas (non-selective).
SI = 1: responds exclusively to one persona (maximally selective).
Left: residual stream. Right: MLP hidden units.
""")

: 

: 

### Figure 3 — Tuning curves of the most persona-selective units

In [ ]:
# Pick the layer with the highest fraction of selective units
best_sel_layer = max(reports, key=lambda l: reports[l]["fraction_selective"])
report = reports[best_sel_layer]
data_t, labels_t, _ = load_layer(ACTIVATIONS_H5, best_sel_layer)

si_scores = report["selectivity_indices"]
top6      = np.argsort(si_scores)[::-1][:6]
cmeans    = report["condition_means"]   # (n_personas, hidden_dim)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for idx, unit in enumerate(top6):
    ax  = axes[idx // 3][idx % 3]
    vals = cmeans[:, unit]
    bars = ax.bar(range(len(persona_short)), vals, color=COLORS, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(persona_short)))
    ax.set_xticklabels(persona_short, rotation=35, ha="right", fontsize=9)
    ax.set_title(f"Unit {unit}  |  SI = {si_scores[unit]:.3f}", fontsize=10)
    ax.set_ylabel("Mean activation" if idx % 3 == 0 else "")
    ax.axhline(0, color="black", lw=0.5); ax.grid(True, alpha=0.3, axis="y")

plt.suptitle(f"Tuning Curves — Top 6 Selective Units\n({best_sel_layer})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig3_tuning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"""
CAPTION — Figure 3
Tuning curves of the 6 most persona-selective units in {best_sel_layer}
(the layer with the highest fraction of significant units).
Each bar shows the mean activation when GPT-2 reads text from one of the
5 interlocutor personas. Units with high SI show a sharp peak at their
preferred persona — analogous to face-selective cells in primate inferotemporal
cortex (Perrett et al. 1982; Quiroga et al. 2005).
""")

: 

: 

## 4 · Population Decoding

A linear (logistic regression) classifier is trained on the full population vector 
of each layer and cross-validated (5-fold stratified) to predict interlocutor persona.  
Accuracy above chance (1/5 = 20 %) means persona identity is **linearly decodable** 
from population activity — a stronger criterion than single-unit selectivity.

In [ ]:
print("Running linear decoding across all layers...")
decoding_results = {}
for layer_name in sorted(reports):
    d, lbl, lm = load_layer(ACTIVATIONS_H5, layer_name)
    res = decode_layer(d, lbl, n_folds=5)
    res["label_map"] = lm
    decoding_results[layer_name] = res
    print(f"  {layer_name:40s}  acc={res['mean_accuracy']:.3f}  "
          f"(chance={res['chance_level']:.2f})")
print("Done.")

: 

: 

### Figure 4 — Linear decoding accuracy across layers

In [ ]:
res_layers_d = sorted([l for l in decoding_results if "residual"   in l])
mlp_layers_d = sorted([l for l in decoding_results if "mlp_hidden" in l])
chance       = decoding_results[res_layers_d[0]]["chance_level"]
x            = range(len(res_layers_d))

acc_res = [decoding_results[l]["mean_accuracy"] for l in res_layers_d]
err_res = [decoding_results[l]["std_accuracy"]   for l in res_layers_d]
acc_mlp = [decoding_results[l]["mean_accuracy"] for l in mlp_layers_d]
err_mlp = [decoding_results[l]["std_accuracy"]   for l in mlp_layers_d]

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(x, [a-e for a,e in zip(acc_res,err_res)],
                   [a+e for a,e in zip(acc_res,err_res)], alpha=0.15, color="steelblue")
ax.plot(x, acc_res, "o-",  color="steelblue", label="Residual stream", lw=2)
ax.fill_between(x, [a-e for a,e in zip(acc_mlp,err_mlp)],
                   [a+e for a,e in zip(acc_mlp,err_mlp)], alpha=0.15, color="coral")
ax.plot(x, acc_mlp, "s--", color="coral",     label="MLP hidden",      lw=2)
ax.axhline(chance, color="gray", ls=":", lw=1.5, label=f"Chance ({chance:.2f})")
ax.set_xlabel("Layer index", fontsize=12)
ax.set_ylabel("Decoding accuracy (5-fold CV)", fontsize=12)
ax.set_title("Linear Decoding of Interlocutor Persona Across Layers", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.05); ax.legend()
ax.set_xticks(x); ax.set_xticklabels(range(len(res_layers_d)))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig4_decoding_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

n_above = sum(1 for r in decoding_results.values() if r["above_chance"])
best_layer_dec = max(decoding_results, key=lambda l: decoding_results[l]["mean_accuracy"])
print(f"""
CAPTION — Figure 4
5-fold cross-validated accuracy of a linear (logistic regression) classifier
trained to decode interlocutor persona from population activity at each layer.
Blue: residual stream. Red: MLP hidden units. Shaded: ±1 SD across folds.
Chance = 1/{len(persona_names)} = {chance:.2f} (uniform over personas).
{n_above}/{len(decoding_results)} layers decode above chance.
Peak: {best_layer_dec} (acc = {decoding_results[best_layer_dec]['mean_accuracy']:.3f}).
Linear decodability implies persona identity is carried in the geometry of the
population code, not just in the firing rates of individual selective units.
""")

: 

: 

### Figure 5 — PCA population geometry at the best-decoding layer

In [ ]:
best_layer_dec = max(decoding_results, key=lambda l: decoding_results[l]["mean_accuracy"])
data_pca, labels_pca, _ = load_layer(ACTIVATIONS_H5, best_layer_dec)
projected, var_ratio     = pca_population_geometry(data_pca, labels_pca)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (d1, d2) in zip(axes, [(0,1),(0,2)]):
    for i, (name, col) in enumerate(zip(persona_short, COLORS)):
        mask = labels_pca == i
        ax.scatter(projected[mask,d1], projected[mask,d2],
                   c=col, label=name, alpha=0.75, s=70,
                   edgecolors="white", linewidths=0.5)
    ax.set_xlabel(f"PC{d1+1} ({var_ratio[d1]*100:.1f}%)", fontsize=11)
    ax.set_ylabel(f"PC{d2+1} ({var_ratio[d2]*100:.1f}%)", fontsize=11)
    ax.legend(title="Persona", fontsize=9); ax.grid(True, alpha=0.3)
axes[0].set_title("PC1 vs PC2", fontsize=12, fontweight="bold")
axes[1].set_title("PC1 vs PC3", fontsize=12, fontweight="bold")
plt.suptitle(f"Population Geometry (PCA) — {best_layer_dec}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig5_pca_geometry.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"""
CAPTION — Figure 5
PCA projection of population activity vectors from {best_layer_dec}
(layer with highest decoding accuracy).
Each point is one interaction episode; color = interlocutor persona.
Clustering of same-color points indicates persona identity is geometrically
separated in the population code.  Variance explained per PC in parentheses.
""")

: 

: 

## 5 · Representational Similarity Analysis (RSA)

An RDM (Representational Dissimilarity Matrix) for each layer is a 5×5 matrix  
where entry (i, j) = correlation distance between the mean population response  
to persona i and persona j.  
Changes in RDM structure across layers reveal how persona representations evolve  
through the network (Kriegeskorte et al. 2008).  

### Figure 6 — RDMs across 6 sampled layers

In [ ]:
n_show     = 6
step       = max(1, len(residual_layers) // n_show)
show_layers = residual_layers[::step][:n_show]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, layer_name in zip(axes, show_layers):
    means = reports[layer_name]["condition_means"]   # (n_personas, hidden_dim)
    rdm   = compute_rdm(means, metric="correlation")
    im    = ax.imshow(rdm, cmap="RdYlBu_r", vmin=0, vmax=rdm.max())
    ax.set_xticks(range(len(persona_short))); ax.set_yticks(range(len(persona_short)))
    ax.set_xticklabels(persona_short, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(persona_short, fontsize=8)
    lshort = (layer_name.replace("block_","L")
                        .replace("_residual","-res")
                        .replace("_mlp_hidden","-mlp"))
    ax.set_title(lshort, fontsize=10, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Representational Dissimilarity Matrices (Correlation Distance)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/fig6_rdms.png", dpi=150, bbox_inches="tight")
plt.show()

print("""
CAPTION — Figure 6
Representational Dissimilarity Matrices (RDMs) for 6 layers (residual stream).
Entry (i,j) = correlation distance between mean population vectors for persona i and j.
0 = identical representations; higher = more distinct.
Warm colors indicate pairs of personas that are more distinguishable at that layer.
Structural change across layers reveals how the network progressively differentiates
(or conflates) interlocutor identities — analogous to RSA in fMRI & electrophysiology
(Kriegeskorte et al. 2008; Yamins & DiCarlo 2016).
""")

: 

: 

## 6 · Summary

In [ ]:
best_dec   = max(decoding_results, key=lambda l: decoding_results[l]["mean_accuracy"])
best_sel   = max(reports,          key=lambda l: reports[l]["fraction_selective"])
n_above    = sum(1 for r in decoding_results.values() if r["above_chance"])

print("=" * 60)
print("  SOCIALLLM — EXPERIMENT SUMMARY")
print("=" * 60)
print(f"  Model            : {MODEL}")
print(f"  Episodes/cond    : {N_EPISODES}")
print(f"  Total interactions: {len(interactions)}")
print()
print(f"  [SELECTIVITY]")
print(f"    Most selective layer : {best_sel}")
print(f"    Fraction selective   : {reports[best_sel]['fraction_selective']:.1%}")
print(f"    Mean SI              : {reports[best_sel]['mean_si']:.3f}")
print(f"    Mean d-prime         : {reports[best_sel]['mean_dprime']:.3f}")
print()
print(f"  [DECODING]")
print(f"    Best layer           : {best_dec}")
print(f"    Peak accuracy        : {decoding_results[best_dec]['mean_accuracy']:.1%}")
print(f"    Chance               : {decoding_results[best_dec]['chance_level']:.1%}")
print(f"    Above-chance layers  : {n_above}/{len(decoding_results)}")
print()
print(f"  Figures saved to: {FIGURES_DIR}/")
for i,name in enumerate(["fig1_fraction_selective","fig2_selectivity_index",
                          "fig3_tuning_curves","fig4_decoding_accuracy",
                          "fig5_pca_geometry","fig6_rdms"],1):
    print(f"    Fig {i}: {name}.png")
print("=" * 60)

: 

: 